# Day 15 · Inside out — แผนที่ที่ "ไม่ถูกต้องพอดี" สามที่

**#30DayMapChallenge 2026 · 15 พ.ย. 2569** (สร้างล่วงหน้าหนึ่งวันในเย็นวันที่ 14) ·
โจทย์: *"Inside out, upside down. Something on this map is not quite right."*

ธีมวันนี้ชวนให้แกล้งวาดแผนที่ผิด แต่ผมเลือกอีกทาง — **กลับกฎสามข้อที่ซีรีส์นี้ใช้อยู่แล้ว**
แล้ววัดของจริงว่าข้อมูลเพี้ยนไปเท่าไร

| การกลับ | กฎที่ถูกกลับ | ผลที่วัดได้ |
|---|---|---|
| ในเป็นนอก | H3 เก็บเซลล์ที่ *จุดกลาง* อยู่ในประเทศ | มีเซลล์ที่ 81.03% ของพื้นที่อยู่นอกประเทศ แต่ยังถูกนับเป็นไทย |
| กลับหัว | หมุนแผนที่ 180° แต่ไม่หมุนพิกัดข้อมูล | 1,737 จาก 1,738 เซลล์ตอบ "จุดที่ใกล้ที่สุด" ผิดจุด |
| กลับทิศคำถาม | เปลี่ยน cell→site เป็น site→cell | 83 จุดเลือกได้แค่ 66 เซลล์ อีก 1,672 เซลล์ไม่มีเจ้าของ |

**รันตามลำดับ cell** · สคริปต์ตรวจตัวเองด้วย `tools/guards.py` ก่อนวาดทุกครั้ง
และเขียนผลลง `day15_stats.json` + `day15_cells_inside_out.csv` ซึ่งเป็นแหล่งตัวเลขเดียวที่โพสต์อ้าง

**ข้อสังเกตที่อยากให้อ่านก่อนรัน** — การหมุน *ทั้งสองฝั่ง* ไม่ใช่ความผิดพลาด บนระนาบ projection
ระยะทางไม่เปลี่ยนแม้แต่มิลิเมตรเดียว (rigid motion) แต่ถ้าวัดระยะทางบนทรงกลมหลังหมุนในระนาบ
จะต่างได้ถึง 0.849 กม. เพราะ "การหมุนแผนที่" กับ "การหมุนโลก" ไม่ใช่การกระทำเดียวกัน

### ค่าคงที่ + ชุดเครื่องมือ (อ่าน/ดาวน์โหลดข้อมูล, ตั้งฟอนต์ไทย, ตรวจว่ารันที่ไหน)

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 15 · Inside out — "Inside out, upside down. Something on this map is not quite right."
(ข้อความหัวข้อจาก 30daymapchallenge.com อ่านสดวันเขียน)

"กลับในเป็นนอก" ในแผ่นนี้ไม่ใช่ลูกเล่นทางภาพ แต่คือกลับ *กฎ* สามข้อที่ซีรีส์นี้ใช้มาตลอด
แล้ววัดว่าข้อมูลเพี้ยนไปเท่าไรจริง ๆ

  1) กฎ "จุดกลางเซลล์อยู่ในประเทศ" (H3 fill)  → เซลล์ที่แทบไม่ได้อยู่ในไทยยังถูกนับเป็นไทย
  2) กลับหัวเฉพาะแผนที่ ไม่กลับข้อมูล        → คำตอบ "จุดทะเบียนที่ใกล้ที่สุด" เพี้ยนกี่กิโลเมตร
  3) กลับทิศของ join (cell→site เป็น site→cell) → ได้คำตอบคนละคำถาม และไม่ใช่การแบ่งพื้นที่

ทุกตัวเลขในโพสต์อ่านจาก day15_stats.json / day15_cells_inside_out.csv ที่สคริปต์นี้เขียนเท่านั้น
"""
import os, sys, json, time, subprocess, urllib.request, zipfile
import numpy as np, pandas as pd, geopandas as gpd
import shapely
from shapely.geometry import Point
from shapely.affinity import rotate as shp_rotate
from shapely.validation import make_valid
from pyproj import CRS, Transformer
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection

T0 = time.time()
COLAB = not os.path.isdir(os.path.join(r"D:\GIS30Days", "geo-notes"))
WORK = os.getcwd() if COLAB else r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "day15_cache") if COLAB else os.path.join(WORK, "cache", "day06")
OUT = os.path.join(WORK, "out", "day15")
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
RAW = "https://raw.githubusercontent.com/thammarat-ai/geo-notes/main/"

def ensure_shp(stem, sub="cultural"):
    shp = os.path.join(CACHE, stem + ".shp")
    if not os.path.exists(shp):
        zp = os.path.join(CACHE, stem + ".zip")
        urllib.request.urlretrieve(f"https://naciscdn.org/naturalearth/10m/{sub}/{stem}.zip", zp)
        zipfile.ZipFile(zp).extractall(CACHE)
    return shp

def repo_file(rel):
    dst = os.path.join(CACHE, os.path.basename(rel))
    if COLAB and not os.path.exists(dst):
        urllib.request.urlretrieve(RAW + rel, dst)
    return dst if COLAB else os.path.join(REPO, rel)

try:
    sys.path.insert(0, os.path.join(WORK, "tools"))
    from guards import (guard_bbox, guard_crs, guard_area, guard_variance,
                        guard_distances, lint_thai, hav_km)
except ModuleNotFoundError:
    gp = os.path.join(CACHE, "guards.py")
    if not os.path.exists(gp):
        urllib.request.urlretrieve(RAW + "tools/guards.py", gp)
    sys.path.insert(0, CACHE)
    from guards import (guard_bbox, guard_crs, guard_area, guard_variance,
                        guard_distances, lint_thai, hav_km)

try:
    import h3
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "h3==4.5.0"])
    import h3

for _f in sorted(os.listdir(CACHE)):
    if _f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(CACHE, _f))
plt.rcParams["font.family"] = "Trirong"
_LOGF = open(os.path.join(OUT, "day15_run.log"), "w", encoding="utf-8")
def log(m):
    s = f"[{time.time()-T0:6.1f}s] {m}"
    print(s, flush=True)
    _LOGF.write(s + chr(10)); _LOGF.flush()

TXT = []
def T(s):
    TXT.append(s)
    return s

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK = "#0d1522", "#e8e3d3"
ACCENT, WARM, COOL, TEAL, MUTED = "#f5b942", "#f4566a", "#38bdf8", "#2dd4bf", "#9fb2c8"
REF_KM2 = 512593.1                     # ตัวเลขที่ซีรีส์ใช้ตั้งแต่ Day 8
DAY12_CLIPPED = 496867.2               # พื้นที่ hex รวมหลัง clip ตามที่ Day 12 บันทึก

## ตอน 1) ข้อมูล + ตรวจขาเข้า

In [ ]:
# ---------------------------------------------------------------- 1. ข้อมูล + ตรวจขาเข้า
ne = gpd.read_file(ensure_shp("ne_10m_admin_0_countries"))
TH = make_valid(ne[ne.NAME == "Thailand"].geometry.iloc[0])
PARTS = list(TH.geoms) if TH.geom_type.startswith("Multi") else [TH]
country_km2 = gpd.GeoSeries([TH], crs="EPSG:4326").to_crs(EQ).area.iloc[0] / 1e6
guard_area(country_km2, REF_KM2, "thailand-area", tol=0.005)
guard_crs(TH.bounds, "thailand-outline", expect="degrees")

df = pd.read_csv(repo_file("data/darksky_thailand.csv"))
df["name"] = df["name"].astype(str).str.strip()
SITES = np.column_stack([df.longitude.values, df.latitude.values])
NAMES = np.array(df["name"].astype(str), dtype=object)   # numpy object array กัน arrow indexing
guard_bbox(SITES, 96.5, 107.6, 4.3, 21.7, "darksky-sites")
assert len(df) == 83, f"ทะเบียนต้องมี 83 จุด ได้ {len(df)}"

g3 = gpd.read_file(repo_file("posts/day03-polygons/day03_hex_polygons.geojson"))
CELLS = g3.geometry.values
assert len(CELLS) == 1738, f"กริด Day 3 ต้องมี 1,738 เซลล์ ได้ {len(CELLS)}"
near_art = np.array([str(x).strip() for x in g3.nearest.values])
prov_art = np.array([str(x).strip() for x in g3.province.values])
cell_km2_full = np.array(gpd.GeoSeries(list(CELLS), crs="EPSG:4326").to_crs(EQ).area) / 1e6
cen_deg = np.array([[c.centroid.x, c.centroid.y] for c in CELLS])
guard_bbox(cen_deg, 96.5, 107.6, 4.3, 21.7, "hex-centroid-degrees")
log(f"ประเทศ {country_km2:,.1f} ตร.กม. · ทะเบียน {len(df)} จุด · กริด {len(CELLS)} เซลล์ "
    f"({cell_km2_full.sum():,.1f} ตร.กม. ถ้าไม่ clip)")

## ตอน 2) การกลับข้อ 1: "inside" ของเซลล์คืออะไรวัดจากอะไร

In [ ]:
# ------------------------------------------ 2. การกลับข้อ 1: "inside" ของเซลล์คืออะไรวัดจากอะไร
t1 = time.perf_counter()
clipped = shapely.intersection(CELLS, TH)
in_km2 = np.array(gpd.GeoSeries(list(clipped), crs="EPSG:4326").to_crs(EQ).area) / 1e6
share = in_km2 / cell_km2_full
assert share.min() > 0, "มีเซลล์ที่ clip แล้วไม่เหลือพื้นที่ = กริดกับเส้นขอบไม่ทับกันเลย"
assert (in_km2 <= cell_km2_full + 1e-6).all(), "พื้นที่ clip ใหญ่กว่าเซลล์เต็ม = มีอะไรพัง"
log(f"clip 1,738 เซลล์กับเส้นขอบ {time.perf_counter()-t1:.1f} วิ")

THRESH = {thr: dict(cells=int((share < thr).sum()),
                    outside_km2=float((cell_km2_full[share < thr] - in_km2[share < thr]).sum()))
          for thr in (0.5, 0.75, 0.9, 0.99, 1.0)}
worst_idx = np.argsort(share)[:6]
WORST = [dict(cell=str(g3.cell.iloc[i]), share_km2=round(float(share[i]), 4),
              outside_pct=round(float(100 * (1 - share[i])), 2),
              inside_km2=round(float(in_km2[i]), 1), full_km2=round(float(cell_km2_full[i]), 1),
              lat=round(float(cen_deg[i][1]), 4), lon=round(float(cen_deg[i][0]), 4),
              province=prov_art[i], nearest_site=near_art[i]) for i in worst_idx]
for w in WORST[:3]:
    log(f"  เซลล์ {w['cell']} เหลือพื้นที่ในไทย {w['share_km2']*100:.1f}% "
        f"({w['inside_km2']:,.1f} จาก {w['full_km2']:,.1f} ตร.กม.) · จังหวัด {w['province']} · "
        f"จุดที่ใกล้สุด {w['nearest_site']}")
guard_variance(share, "inside-share", min_spread_km=0)

# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 2b. complement: ตารางที่ "ไม่ใช่ประเทศไทย" ในกล่องคลุมประเทศ (ใช้ H3 เหมือน Day 14)
lo, la, hi, la_hi = TH.bounds
bbox_poly = h3.LatLngPoly([(la, lo), (la_hi, lo), (la_hi, hi), (la, hi)])
bbox_cells = set(h3.polygon_to_cells(bbox_poly, 5))
th_cells = set(str(c) for c in g3.cell.values)          # ใช้ H3 index ที่ Day 3 บันทึกไว้โดยตรง
assert th_cells <= bbox_cells, "เซลล์ของ Day 3 ต้องอยู่ในกล่องคลุมประเทศทั้งหมด"
COMPL = dict(bbox_cells=len(bbox_cells), inside_cells=len(th_cells & bbox_cells),
             outside_cells=len(bbox_cells - th_cells),
             bbox_area_km2=float((gpd.GeoSeries([shapely.box(lo, la, hi, la_hi)], crs="EPSG:4326")
                                  .to_crs(EQ).area.iloc[0] / 1e6)))
COMPL["country_share_of_bbox_pct"] = 100 * country_km2 / COMPL["bbox_area_km2"]
log(f"กล่องคลุมประเทศมี res 5 จำนวน {COMPL['bbox_cells']:,} เซลล์ · "
    f"ที่อยู่ในไทย {COMPL['inside_cells']:,} · นอกไทย {COMPL['outside_cells']:,} · "
    f"ประเทศกินพื้นที่กล่องเพียง {COMPL['country_share_of_bbox_pct']:.1f}%")

## ตอน 2c) cross-check ตัวเลขของ Day 7 ที่เคยหาที่มาไม่ได้

In [ ]:
# ------------------------------------- 2c. cross-check ตัวเลขของ Day 7 ที่เคยหาที่มาไม่ได้
XDAY7 = dict(country_not_covered_km2=float(country_km2 - in_km2.sum()),
             grid_area_outside_country_km2=float(cell_km2_full.sum() - in_km2.sum()),
             day07_post_says_not_covered=18692.0, day07_post_says_overflow=2967.0)
XDAY7["delta_not_covered"] = round(XDAY7["country_not_covered_km2"] - 18692.0, 1)
XDAY7["delta_overflow"] = round(XDAY7["grid_area_outside_country_km2"] - 2967.0, 1)
log(f"Day 7 บอก hex กลบไม่ถึง 18,692 ตร.กม. · วันนี้คำนวณได้ "
    f"{XDAY7['country_not_covered_km2']:,.1f} (ต่าง {XDAY7['delta_not_covered']:+.1f}) และ "
    f"ล้นออกนอกเส้น 2,967 · วันนี้ {XDAY7['grid_area_outside_country_km2']:,.1f} "
    f"(ต่าง {XDAY7['delta_overflow']:+.1f})")

## ตอน 3) การกลับข้อ 2: กลับหัวเฉพาะแผนที่ ไม่กลับข้อมูล

In [ ]:
# ------------------------------------------- 3. การกลับข้อ 2: กลับหัวเฉพาะแผนที่ ไม่กลับข้อมูล
_tr = Transformer.from_crs("EPSG:4326", EQ, always_xy=True)
_c = gpd.GeoSeries([TH], crs="EPSG:4326").to_crs(EQ).centroid.iloc[0]
cx, cy = _c.x, _c.y
cen_m = np.array(_tr.transform(cen_deg[:, 0], cen_deg[:, 1])).T
rot_m = np.column_stack([2 * cx - cen_m[:, 0], 2 * cy - cen_m[:, 1]])      # หมุน 180° รอบ centroid
rot_deg = np.array(_tr.transform(rot_m[:, 0], rot_m[:, 1], direction="INVERSE")).T
guard_crs((rot_deg[:, 0].min(), rot_deg[:, 1].min(), rot_deg[:, 0].max(), rot_deg[:, 1].max()),
          "rotated-cell-centres", expect="degrees")

D_true = hav_km(cen_deg, SITES)
guard_distances(D_true, "cell-to-site-true", max_km=2500)
ans_true = NAMES[D_true.argmin(axis=1)]
agree_art = float(np.mean(ans_true == near_art))
d_true = D_true.min(axis=1)

D_flip = hav_km(rot_deg, SITES)          # แผนที่กลับหัวแล้ว แต่จุดทะเบียนยังอยู่พิกัดเดิม
guard_distances(D_flip, "cell-to-site-flipped", max_km=2500)
ans_flip = NAMES[D_flip.argmin(axis=1)]
d_flip = D_flip.min(axis=1)
extra = d_flip - d_true
changed = int((ans_flip != ans_true).sum())
guard_variance(extra, "flip-extra-km", min_spread_km=1.0)
log(f"ตรวจซ้ำกับไฟล์ Day 3: คำตอบ nearest ของวันนี้ตรงกับที่ไฟล์บันทึก {100*agree_art:.2f}% "
    f"({int((ans_true != near_art).sum())} เซลล์ต่าง — centroid ของ polygon ≠ จุดกลางของ H3 cell ที่ถูก clip)")
log(f"กลับหัวเฉพาะแผนที่: {changed:,} จาก {len(CELLS):,} เซลล์ตอบผิดจุด "
    f"({100*changed/len(CELLS):.1f}%) · ระยะที่ไกลขึ้น ค่าต่ำสุด {extra.min():,.2f} · "
    f"ค่ามัธยฐาน {np.median(extra):,.2f} · ค่าเฉลี่ย {extra.mean():,.2f} · "
    f"ค่าสูงสุด {extra.max():,.2f} กม.")

# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 3b. การหมุนไม่เปลี่ยนข้อมูล (ยืนยันด้วยการคำนวณ ไม่ใช่ด้วยการพูด)
s_m = np.array(_tr.transform(SITES[:, 0], SITES[:, 1])).T
s_rot = np.column_stack([2 * cx - s_m[:, 0], 2 * cy - s_m[:, 1]])
s_rot_deg = np.array(_tr.transform(s_rot[:, 0], s_rot[:, 1], direction="INVERSE")).T
D_rot = hav_km(rot_deg, s_rot_deg)          # หมุนทั้งแผนที่และข้อมูลพร้อมกัน (บนระนาบ projection)
guard_distances(D_rot[:2000], "cell-to-site-rotated-both", max_km=2500)
rot_both_diff = np.abs(D_rot.min(axis=1) - d_true)
# บนระนาบที่หมุน การหมุน 180° เป็น rigid motion → ระยะทางต้องไม่เปลี่ยนเลย
d_planar_true = np.sqrt(((cen_m[:, 0][:, None] - s_m[:, 0][None, :])**2 +
                        (cen_m[:, 1][:, None] - s_m[:, 1][None, :])**2)).min(axis=1)
d_planar_rot = np.sqrt(((rot_m[:, 0][:, None] - s_rot[:, 0][None, :])**2 +
                       (rot_m[:, 1][:, None] - s_rot[:, 1][None, :])**2)).min(axis=1)
planar_diff_m = float(np.abs(d_planar_rot - d_planar_true).max())
FLIP = dict(rotation_degrees=180,
            rotation_origin="centroid ของประเทศไทยใน Equidistant Conic (หน่วยเมตร)",
            changed_cells=changed, changed_pct=round(100 * changed / len(CELLS), 2),
            extra_km_min=round(float(extra.min()), 3), extra_km_median=round(float(np.median(extra)), 3),
            extra_km_mean=round(float(extra.mean()), 3), extra_km_max=round(float(extra.max()), 3),
            distance_true_mean_km=round(float(d_true.mean()), 3),
            distance_flipped_mean_km=round(float(d_flip.mean()), 3),
            agree_with_day3_artifact_pct=round(100 * agree_art, 3),
            rotate_both_geodesic_max_abs_diff_km=round(float(rot_both_diff.max()), 3),
            rotate_both_planar_max_abs_diff_m=round(planar_diff_m, 6))
log(f"ถ้าหมุนทั้งแผนที่และข้อมูลพร้อมกัน: บนระนาบ projection ระยะทางไม่เปลี่ยนเลย "
    f"(ต่างสูงสุด {planar_diff_m:.6f} ม.) แต่บนทรงกลมจริงต่างได้ถึง "
    f"{rot_both_diff.max():,.3f} กม. เพราะการหมุนใน projection ไม่ใช่การหมุนบนโลก — "
    f"การกลับหัวไม่ใช่ปัญหา ปัญหาคือการกลับข้างเดียว")

## ตอน 4) การกลับข้อ 3: join จากฝั่งจุด (site → cell)

In [ ]:
# ------------------------------------ 4. การกลับข้อ 3: join จากฝั่งจุด (site → cell)
own_answer = lambda c: str(near_art[c])                   # คำตอบที่ซีรีส์บันทึกไว้สำหรับเซลล์ c
# D_true เป็นเมทริกซ์ (เซลล์ × จุด) — เริ่มจากฝั่งจุดต้อง argmin ตาม axis=0 (ได้ยาว 83)
site_to_cell = D_true.argmin(axis=0)                      # แต่ละจุดเลือกเซลล์ที่ใกล้ตัวมันที่สุด
assert site_to_cell.size == len(df), "site→cell ต้องมี 83 ค่า (หนึ่งค่าต่อหนึ่งจุด)"
cells_claimed = np.unique(site_to_cell)
mutual = np.zeros(len(CELLS), bool)
for k, c in enumerate(site_to_cell):
    if ans_true[c] == NAMES[k]:                            # เซลล์นั้นก็ต้องเลือกจุดเดียวกันกลับ
        mutual[c] = True
farther = []
for k in range(len(df)):
    c = int(site_to_cell[k])
    if own_answer(c) != NAMES[k]:
        farther.append(D_true[c, k] - d_true[c])            # ผู้ใช้เซลล์นั้นต้องเดินทางไกลขึ้นเท่าไร
farther = np.array(farther) if farther else np.array([0.0])
JOIN = dict(sites=len(df), cells=len(CELLS), cells_claimed_by_a_site=int(len(cells_claimed)),
            cells_never_claimed=int(len(CELLS) - len(cells_claimed)),
            sites_sharing_a_cell=int(len(df) - len(cells_claimed)),
            mutual_pairs=int(mutual.sum()), non_mutual_cells=int(len(cells_claimed) - mutual.sum()),
)
log(f"join กลับทิศ: {len(df)} จุดเลือกได้ {JOIN['cells_claimed_by_a_site']} เซลล์ "
    f"(เหลือ {JOIN['cells_never_claimed']} เซลล์ไม่มีจุดมาเลือก) · คู่ที่เลือกซึ่งกันและกัน "
    f"{JOIN['mutual_pairs']} คู่")

# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 4b. ถ้ายึด "เซลล์ที่จุดเลือก" เป็นคำตอบ ผู้ใช้เซลล์นั้นจะไกลขึ้นกี่กิโลเมตร
assigned = NAMES.copy()                                   # จุด k เลือกเซลล์ site_to_cell[k]
own = np.array([own_answer(int(c)) for c in site_to_cell], dtype=object)
diff_class = int((assigned != own).sum())
JOIN["claimed_cell_answer_differs"] = diff_class
JOIN["claimed_cell_extra_km_mean"] = round(float(farther.mean()), 3)
JOIN["claimed_cell_extra_km_max"] = round(float(farther.max()), 3)
log(f"ใน {len(df)} เซลล์ที่จุดมาเลือก เซลล์นั้นตอบผิดจากจุดที่เลือกมัน {diff_class} ครั้ง "
    f"(ไกลขึ้นเฉลี่ย {farther.mean():,.2f} กม. · สูงสุด {farther.max():,.2f} กม.)")

## ตอน 5) ภาพ 1 — แผ่นหลักสามช่อง

In [ ]:
# ------------------------------------------------ 5. ภาพ 1 — แผ่นหลักสามช่อง
def rings_of(geoms):
    """แปลง Polygon/MultiPolygon เป็น list ของ ring (กัน .exterior พังกับ MultiPolygon)"""
    out = []
    for g in geoms:
        for q in (g.geoms if g.geom_type.startswith("Multi") else [g]):
            out.append(list(q.exterior.coords))
    return out

def hex_outline(c, nd=5):
    return [(round(float(p[1]), nd), round(float(p[0]), nd)) for p in h3.cell_to_boundary(c)]

def draw_hero():
    fig = plt.figure(figsize=(16.4, 6.4), dpi=150)
    fig.patch.set_facecolor(BG)
    gs = fig.add_gridspec(1, 3, left=0.015, right=0.985, wspace=0.06)

    # (a) ปกติ
    ax = fig.add_subplot(gs[0, 0]); ax.set_facecolor(BG)
    ax.add_collection(PolyCollection(rings_of(CELLS), facecolor="#2c4a63", edgecolor=BG, lw=0.25))
    for p in PARTS:
        x, y = p.exterior.xy
        ax.plot(x, y, color=INK, lw=0.8, zorder=4)
    ax.scatter(SITES[:, 0], SITES[:, 1], s=18, color="#0d1522", edgecolor=ACCENT, lw=0.8, zorder=6)
    ax.set_title(T("(ก) ตามปกติ\n1,738 เซลล์ที่จุดกลางอยู่ในประเทศ"), color=INK, fontsize=11.5)

    # (b) กลับในเป็นนอก: เซลล์นอกประเทศถูก tô ทับ ประเทศกลายเป็นรู
    ax = fig.add_subplot(gs[0, 1]); ax.set_facecolor(BG)
    out_cells = sorted(bbox_cells - th_cells)
    rings_out = [hex_outline(c) for c in out_cells]
    ax.add_collection(PolyCollection(rings_out, facecolor="#5a3f52", edgecolor=BG, lw=0.2))
    ax.add_collection(PolyCollection(rings_of(CELLS), facecolor="none", edgecolor=COOL, lw=0.3))
    for p in PARTS:
        x, y = p.exterior.xy
        ax.plot(x, y, color=INK, lw=0.9, zorder=4)
    bad = [hex_outline(str(g3.cell.iloc[i])) for i in worst_idx[:3]]
    ax.add_collection(PolyCollection(bad, facecolor="none", edgecolor=WARM, lw=1.8, zorder=6))
    ax.set_title(T(f"(ข) กลับในเป็นนอก\nเซลล์นอกประเทศ {len(out_cells):,} ก้อน · "
                   f"3 เซลล์สีส้มคือเซลล์ที่ <50% เป็นไทย"), color=INK, fontsize=11.5)

    # (c) กลับหัวเฉพาะแผนที่
    ax = fig.add_subplot(gs[0, 2]); ax.set_facecolor(BG)
    cells_m = gpd.GeoSeries(list(CELLS), crs="EPSG:4326").to_crs(EQ).geometry.values
    rot_m = [shp_rotate(g, 180, origin=(cx, cy)) for g in cells_m]
    rot_deg = gpd.GeoSeries(list(rot_m), crs=EQ).to_crs("EPSG:4326").geometry.values
    ax.add_collection(PolyCollection(rings_of(rot_deg),
                                     facecolor="#2c4a63", edgecolor=BG, lw=0.25))
    for p in PARTS:
        x, y = p.exterior.xy
        ax.plot(x, y, color=INK, lw=0.8, alpha=0.35, zorder=3)
    ax.scatter(SITES[:, 0], SITES[:, 1], s=18, color="#0d1522", edgecolor=WARM, lw=0.8, zorder=6)
    ax.annotate(T("↑ เหนือยังอยู่ข้างบน\nแต่ประเทศถูกหมุนไปแล้ว 180°"), (106.1, 20.9),
                color=WARM, fontsize=10.5, ha="right", va="top", linespacing=1.4)
    ax.set_title(T(f"(ค) กลับหัวเฉพาะแผนที่\n{changed:,} เซลล์ตอบผิดจุด "
                   f"({100*changed/len(CELLS):.1f}%)"), color=INK, fontsize=11.5)

    for a in fig.axes:
        a.set_xlim(97.2, 106.4); a.set_ylim(4.8, 21.2); a.set_aspect("equal"); a.axis("off")
    fig.text(0.5, 0.015, T("จุดส้มในช่อง (ค) คือจุดทะเบียนตำแหน่งจริง — แผนที่หมุนแต่ข้อมูลไม่หมุน "
                           "ผลลัพธ์ไม่ใช่ภาพที่ดูแปลก แต่คือคำตอบที่ผิดเป็นกิโลเมตร"),
             color=MUTED, fontsize=10.5, ha="center")
    fig.suptitle(T("กลับในเป็นนอก · กลับหัว · กลับทิศของ join — สามอย่างนี้ไม่ใช่ลูกเล่น "
                   "แต่คือกฎที่เราไม่ได้เขียนไว้"), color=INK, fontsize=15.5, y=0.99)
    p = os.path.join(OUT, "day15_inside_out.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

## ตอน 6) ภาพ 2 — เซลล์ "เป็นไทย" กี่เปอร์เซ็นต์

In [ ]:
# ------------------------------------------------ 6. ภาพ 2 — เซลล์ "เป็นไทย" กี่เปอร์เซ็นต์
def draw_share():
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14.4, 5.4), dpi=150,
                                  gridspec_kw=dict(width_ratios=[1.15, 1]))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor("#111c2b")
    bins = np.linspace(0, 1.0, 41)
    ax.hist(share * 100, bins=bins * 100, color=TEAL, edgecolor=BG, lw=0.4)
    ax.axvline(50, color=WARM, ls="--", lw=1.4)
    ax.text(51, ax.get_ylim()[1] * 0.82, T("เกณฑ์ 50%"), color=WARM, fontsize=10)
    ax.set_xlabel(T("สัดส่วนพื้นที่เซลล์ที่อยู่ในประเทศไทย (ร้อยละ)"))
    ax.set_ylabel(T("จำนวนเซลล์ (มาตราลอการิทึม)"))
    ax.set_yscale("log")
    ax.set_title(T("เซลล์ที่ระบบนับว่าเป็นไทย เป็นไทยจริงแค่ไหน"), color=INK, fontsize=12.5)
    ax.tick_params(colors=INK, labelsize=10)
    for s in ax.spines.values(): s.set_color("#33415e")
    for s in ("top", "right"): ax.spines[s].set_visible(False)

    ax2.set_facecolor("#111c2b"); ax2.axis("off")
    rows = [(f"{int(k*100)}% หรือน้อยกว่า", v["cells"], f"{v['outside_km2']:,.1f} ตร.กม.")
            for k, v in THRESH.items() if k < 1.0]
    ax2.text(0.02, 0.95, T("เซลล์ที่พื้นที่เกินเกณฑ์น้อยว่า..."), color=INK, fontsize=11.5)
    ax2.text(0.62, 0.95, T("จำนวน"), color=INK, fontsize=11.5)
    ax2.text(0.78, 0.95, T("ส่วนที่ล้นออกนอกประเทศ"), color=INK, fontsize=11.5)
    y = 0.87
    for lab, n, area in rows:
        ax2.text(0.02, y, lab, color="#cfe0f0", fontsize=11)
        ax2.text(0.62, y, f"{n:,}", color=ACCENT, fontsize=11)
        ax2.text(0.78, y, area, color=TEAL, fontsize=11)
        y -= 0.075
    y -= 0.03
    ax2.text(0.02, y, T("หกเซลล์ที่ 'เป็นไทย' น้อยที่สุด (ชื่อจากไฟล์ ไม่ใช่จากภาพ)"),
             color=INK, fontsize=11.5); y -= 0.07
    for w in WORST[:6]:
        ax2.text(0.02, y, f"{w['share_km2']*100:5.1f}%  {w['province']} · {w['nearest_site']}",
                 color="#cfe0f0", fontsize=10.5)
        y -= 0.062
    ax2.text(0.02, 0.03, T(f"ค่ามัธยฐาน {np.median(share)*100:.1f}% · ค่าเฉลี่ย {share.mean()*100:.2f}% "
                           f"· ค่าต่ำสุด {share.min()*100:.1f}%"), color=MUTED, fontsize=10.5)
    fig.suptitle(T("กฎ 'จุดกลางอยู่ในประเทศ' ไม่ได้บอกว่าเซลล์ส่วนใหญ่อยู่ในประเทศ"),
                 color=INK, fontsize=15, y=0.99)
    p = os.path.join(OUT, "day15_inside_share.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

## ตอน 7) ภาพ 3 — กลับหัว กับ กลับทิศของ join

In [ ]:
# ------------------------------------------------ 7. ภาพ 3 — กลับหัว กับ กลับทิศของ join
def draw_join():
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14.4, 5.4), dpi=150)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor("#111c2b")
    ax.hist(np.clip(extra, 0, None), bins=np.linspace(0, min(extra.max(), 900), 46),
            color=WARM, edgecolor=BG, lw=0.4)
    ax.axvline(extra.mean(), color=INK, ls="--", lw=1.3)
    ax.text(extra.mean() * 1.03, ax.get_ylim()[1] * 0.86,
            T(f"ค่าเฉลี่ย {extra.mean():,.1f} กม."), color=INK, fontsize=10)
    ax.text(0.02, 0.62, T(f"บางเซลล์กลับใกล้ขึ้น: ค่าต่ำสุด {extra.min():,.1f} กม." + chr(10) +
                          f"แต่ส่วนใหญ่ไกลขึ้น · ค่ามัธยฐาน {np.median(extra):,.1f} กม."),
            transform=ax.transAxes, color=MUTED, fontsize=10, va="top", linespacing=1.5)
    ax.set_xlabel(T("ระยะที่ไกลขึ้นเมื่อแผนที่ถูกกลับหัวแต่ข้อมูลไม่กลับ (กิโลเมตร)"))
    ax.set_ylabel(T("จำนวนเซลล์"))
    ax.set_title(T(f"กลับหัวเฉพาะแผนที่: {changed:,} เซลล์ตอบผิดจุด "
                   f"(สูงสุด {extra.max():,.0f} กม.)"), color=INK, fontsize=12)
    ax.tick_params(colors=INK, labelsize=10)
    for s in ax.spines.values(): s.set_color("#33415e")
    for s in ("top", "right"): ax.spines[s].set_visible(False)

    ax2.set_facecolor("#111c2b")
    cats = [T("จุด 83 จุด\nเลือกเซลล์"), T("เซลล์ที่ได้เจ้าของ"), T("คู่ที่เลือก\nซึ่งกันและกัน"),
            T("เซลล์ที่\nไม่มีจุดมาเลือก")]
    vals = [83, JOIN["cells_claimed_by_a_site"], JOIN["mutual_pairs"], JOIN["cells_never_claimed"]]
    cols = [ACCENT, TEAL, COOL, WARM]
    bars = ax2.bar(range(4), vals, color=cols, width=0.62)
    for b, v in zip(bars, vals):
        ax2.text(b.get_x() + b.get_width()/2, v * 1.02, f"{v:,}", color=INK, ha="center", fontsize=11)
    ax2.set_xticks(range(4)); ax2.set_xticklabels(cats, color=INK, fontsize=10.5)
    ax2.set_yscale("log"); ax2.set_ylim(1, 4000)
    ax2.tick_params(colors=INK, labelsize=10)
    ax2.set_title(T(f"join กลับทิศตอบอีกคำถาม: {len(CELLS):,} เซลล์เหลือ "
                    f"{JOIN['cells_claimed_by_a_site']} เซลล์ที่มีเจ้าของ"),
                  color=INK, fontsize=12)
    for s in ax2.spines.values(): s.set_color("#33415e")
    for s in ("top", "right"): ax2.spines[s].set_visible(False)
    fig.suptitle(T("กลับหัวไม่ใช่ความผิดพลาด ถ้ากลับทั้งสองฝั่ง · "
                   "ที่ผิดพลาดจริงคือการกลับข้างเดียว และเปลี่ยนทิศของคำถาม"),
                 color=INK, fontsize=14.5, y=0.99)
    p = os.path.join(OUT, "day15_upside_down.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

P1, P2, P3 = draw_hero(), draw_share(), draw_join()

## ตอน 8) artifact

In [ ]:
# ------------------------------------------------ 8. artifact
def pyo(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"เข้ารหัส JSON ไม่ได้ที่ {type(o)}")

pd.DataFrame(dict(cell=g3.cell.values, province=prov_art, nearest_site=near_art,
                  full_km2=np.round(cell_km2_full, 3), inside_km2=np.round(in_km2, 3),
                  inside_share=np.round(share, 5),
                  answer_if_map_flipped=ans_flip)).to_csv(os.path.join(OUT, "day15_cells_inside_out.csv"),
                                                          index=False)
STATS = dict(
    theme="Inside out, upside down. Something on this map is not quite right.",
    theme_source="30daymapchallenge.com หัวข้อ Day 15 'Inside out' (อ่านสดวันเขียน)",
    generated_local=time.strftime("%Y-%m-%d %H:%M:%S"),
    made_on="14 พ.ย. 2569 (ทำล่วงหน้าหนึ่งวัน · ลงวันที่ 15 ตามโจทย์)",
    h3_version=h3.__version__,
    inputs=dict(hex_grid_cells=int(len(CELLS)), site_register=int(len(df)),
                outline_source="Natural Earth 10m admin_0_countries",
                country_area_km2=round(country_km2, 1)),
    inside_share=dict(worst_cell_outside_pct=round(float(100 * (1 - share.min())), 2),
                      cells_fully_inside=int((share >= 0.999999).sum()),
                      mean_pct=round(float(share.mean()*100), 3),
                      median_pct=round(float(np.median(share)*100), 3),
                      min_pct=round(float(share.min()*100), 3),
                      max_pct=round(float(share.max()*100), 3),
                      thresholds=THRESH, worst_six=WORST,
                      sum_inside_km2=round(float(in_km2.sum()), 1),
                      sum_full_km2=round(float(cell_km2_full.sum()), 1)),
    complement_of_bbox=COMPL,
    upside_down=FLIP,
    join_inversion=JOIN,
    cross_check_day07=XDAY7,
    cross_check_day14=dict(day14_res5_cells_ne10m=1745, day3_grid_cells=int(len(CELLS)),
                           day14_sites_without_own_cell=5,
                           day14_sites_outside_outline=4,
                           note="Day 14 นับจากขอบ NE 10m ด้วย H3 fill วันนี้ใช้กริดของ Day 3 "
                                "(GADM) มา clip กับขอบ NE 10m — สองวิธีให้เลขคนละชุดแต่เรื่องเดียวกัน"),
    figures=["day15_inside_out.png", "day15_inside_share.png", "day15_upside_down.png"],
    metric_definitions={
        "worst_cell_outside_pct": "ร้อยละของพื้นที่เซลล์ที่อยู่นอกประเทศ สำหรับเซลล์ที่ 'เป็นไทย' น้อยที่สุด",
        "cells_fully_inside": "จำนวนเซลล์ที่ clip แล้วพื้นที่เท่ากับเซลล์เต็มทุกตารางเมตร (share = 100%)",
        "inside_share": "พื้นที่เซลล์หลัง clip กับเส้นขอบประเทศ หารด้วยพื้นที่เซลล์เต็ม (ร้อยละ) — "
                        "บอกว่า 'เซลล์ที่ระบบนับเป็นไทย' เป็นไทยจริงเท่าไร",
        "thresholds": "จำนวนเซลล์ที่ inside_share ต่ำกว่าเกณฑ์ พร้อมผลรวมพื้นที่ที่ล้นออกนอกประเทศ (ตร.กม.)",
        "bbox_cells": "จำนวนเซลล์ H3 res 5 ที่จุดกลางอยู่ในกล่องคลุมประเทศ (รวมส่วนที่เป็นประเทศเพื่อนบ้าน)",
        "outside_cells": "เซลล์ในกล่องคลุมประเทศที่จุดกลางอยู่นอกไทย = complement ที่วาดในช่อง (ข)",
        "country_share_of_bbox_pct": "พื้นที่ประเทศ ÷ พื้นที่กล่องคลุมประเทศ × 100 — 'ความเปล่า' โดยรอบ",
        "changed_cells": "จำนวนเซลล์ที่ตอบ 'จุดทะเบียนที่ใกล้ที่สุด' เปลี่ยนไปเมื่อหมุนแผนที่ 180° "
                         "รอบ centroid ของประเทศ แต่ไม่หมุนพิกัดจุดทะเบียน",
        "extra_km_mean/max": "ระยะทางสู่จุดที่ใกล้ที่สุดที่เพิ่มขึ้นเพราะกลับหัวเฉพาะแผนที่ "
                             "(มัธยฐาน/เฉลี่ย/สูงสุด เป็นกิโลเมตร)",
        "rotate_both_planar_max_abs_diff_m": "ผลต่างสูงสุดของระยะทาง (เมตร) เมื่อหมุนทั้งแผนที่และข้อมูล 180° บนระนาบ projection แล้ววัดระยะบนระนาบเดียวกัน — ต้องเป็น 0 เพราะการหมุนคือ rigid motion",
        "rotate_both_geodesic_max_abs_diff_km": "ผลต่างสูงสุดของระยะทาง (กิโลเมตร) เมื่อหมุนทั้งแผนที่และข้อมูลบนระนาบ projection แต่ไปวัดระยะบนทรงกลม — ไม่เป็น 0 เพราะการหมุนในแผนที่ไม่ใช่การหมุนบนโลก",
        "cells_claimed_by_a_site": "จำนวนเซลล์ที่แตกต่างกันที่จุด 83 จุดเลือกเมื่อเริ่มจากฝั่งจุด (site→cell)",
        "mutual_pairs": "จำนวนคู่ที่เซลล์เลือกจุดนั้น และจุดนั้นก็เลือกเซลล์นั้น (mutual nearest neighbour)",
        "cells_never_claimed": "เซลล์ที่ไม่มีจุดทะเบียนใดเลือกเลยเมื่อเริ่มจากฝั่งจุด",
        "claimed_cell_extra_km_mean": "ผลเฉลี่ยของ 'ถ้าเชื่อการเลือกจากฝั่งจุด' ผู้ใช้เซลล์นั้นจะไกลขึ้นกี่กิโลเมตร",
        "cross_check_day07": "ตัวเลขในโพสต์ Day 7 ที่เคยหาที่มาไม่ได้ใน artifact เทียบกับที่คำนวณใหม่วันนี้"},
    note="ภาพช่อง (ค) ตั้งใจให้ประเทศกลับหัวเพื่อสาธิตความผิดพลาดที่วัดได้ ไม่ใช่แผนที่สำหรับใช้งาน")
json.dump(STATS, open(os.path.join(OUT, "day15_stats.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, default=pyo)
log(f"เขียน day15_stats.json + day15_cells_inside_out.csv ที่ {OUT}")

## ตอน 9) referee

In [ ]:
# ------------------------------------------------ 9. referee
for f in ["day15_inside_out.png", "day15_inside_share.png", "day15_upside_down.png",
          "day15_cells_inside_out.csv", "day15_stats.json"]:
    pth = os.path.join(OUT, f)
    assert os.path.exists(pth) and os.path.getsize(pth) > 0, f"ไฟล์ขาดหรือว่าง: {f}"
assert not [f for f in os.listdir(OUT) if f.startswith("_tmp")], "ต้องลบไฟล์ชั่วคราวให้หมด"
json.load(open(os.path.join(OUT, "day15_stats.json"), encoding="utf-8"))
lint_thai(" ".join(TXT) + " " + __doc__, "day15-figure-texts")
_src = os.path.abspath(__file__) if "__file__" in globals() else None
if _src and os.path.exists(_src):
    _bad = []
    for _n, _l in enumerate(open(_src, encoding="utf-8").read().split(chr(10)), 1):
        if any(0x0E00 <= ord(c) <= 0x0E7F for c in _l):
            for c in _l:
                o = ord(c)
                if (0x0100 <= o <= 0x017F) or (0x1E00 <= o <= 0x1EFF) or \
                   (0x3000 <= o <= 0x30FF) or (0xAC00 <= o <= 0xD7AF) or (0x4E00 <= o <= 0x9FFF):
                    _bad.append((_n, c))
    assert not _bad, f"พบอักษรไม่ใช่ไทย/ASCII ในบรรทัดที่มีภาษาไทย: {_bad[:6]}"
    log("ตรวจ unicode ทั้งไฟล์แล้ว")
assert abs(share.mean() - 0.9926) < 0.01, "ค่าเฉลี่ย inside share เปลี่ยนจากที่ตรวจไว้ — ทบทวนเส้นขอบ"
assert planar_diff_m < 1e-3, "หมุนทั้งสองฝั่งบนระนาบแล้วระยะทางต้องไม่เปลี่ยน (rigid motion)"
log("สรุป: " + json.dumps(dict(
    cells=len(CELLS), median_share_pct=round(float(np.median(share)*100), 2),
    min_share_pct=round(float(share.min()*100), 2), cells_below_50=THRESH[0.5]["cells"],
    outside_km2=round(float(cell_km2_full.sum() - in_km2.sum()), 1),
    uncovered_km2=round(float(country_km2 - in_km2.sum()), 1),
    flipped_changed_cells=changed, flip_extra_mean_km=FLIP["extra_km_mean"],
    flip_extra_max_km=FLIP["extra_km_max"],
    rotate_planar_diff_m=round(planar_diff_m, 6),
    rotate_geodesic_diff_km=FLIP["rotate_both_geodesic_max_abs_diff_km"],
    claimed_cells=JOIN["cells_claimed_by_a_site"], mutual=JOIN["mutual_pairs"],
    never_claimed=JOIN["cells_never_claimed"]), ensure_ascii=False))
print("DONE day15 ใน %.1f วิ" % (time.time() - T0))
_LOGF.close()